# Grounded-rationale SFT for Qwen2.5-0.5B

This notebook adapts the existing answer-only BioASQ SFT adapter to the response format required by the planned answer-plus-rationale DPO experiment.

It deliberately **does not call an annotation API**. It uses the 1,111 accepted GPT-4.1-mini gold rationales already stored on disk, adds a deterministic answer-only replay subset, and continues the existing LoRA adapter with a small learning rate.

The rationale response contract is:

```text
Reason: <two evidence-grounded sentences>
Answer: [BE]<one extractive answer>[EE]
```

The prompt shows numbered snippets because the accepted rationales cite identifiers such as `1.1`. The final answer remains machine-readable for official BioASQ scoring.

In [ ]:
from pathlib import Path
import sys, json, re, hashlib, random, math
from collections import Counter


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "cse_dpo").is_dir() and (path / "models").is_dir():
            return path
    raise FileNotFoundError("Run this notebook from inside the Task-Structured Counterfactual Preference Mining project.")


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Inputs
BASE_MODEL = ROOT / "models/Qwen2.5-0.5B-Instruct"
INITIAL_SFT_ADAPTER = (
    ROOT / "Artifacts/Factoid_SFT/models/"
    "evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005_strict_extractive/"
    "adapter_best_evidence_mrr"
)
RATIONALE_BANK = (
    ROOT / "Artifacts/cse_dpo/gold_answer_rationales/"
    "gpt41mini_gold_supported_1130_v2/gold_answer_rationales.jsonl"
)
SOURCE_TRAIN = (
    ROOT / "data/BioASQ_factoid_sft_prepared/"
    "evidence_grounded_single_answer_qwen25_05b/train_prepared.json"
)
DEV_INPUT = (
    ROOT / "data/BioASQ_factoid_sft_prepared/"
    "single_answer_full_resources_qwen25_05b/eval_prepared.json"
)
PROMPT_FILE = ROOT / "prompts/factoid_grounded_rationale_sft.json"

# A new directory prevents overwriting the answer-only SFT model.
RUN_NAME = "qwen25_05b_grounded_rationale_sft_positive1111_replay20_v1"
OUTPUT_DIR = ROOT / "Artifacts/Factoid_SFT/models" / RUN_NAME
DATA_DIR = OUTPUT_DIR / "prepared_data"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FINAL_ADAPTER_DIR = OUTPUT_DIR / "adapter_final"

# Conservative format-adaptation settings.
SEED = 3407
ANSWER_ONLY_REPLAY_FRACTION = 0.20  # fraction of the final mixed dataset
MAX_SEQ_LENGTH = 4096
MAX_NEW_TOKENS = 192
NUM_TRAIN_EPOCHS = 1.0
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 32
SAVE_STEPS = 25
LOAD_IN_4BIT = True

for path in (BASE_MODEL, INITIAL_SFT_ADAPTER, RATIONALE_BANK, SOURCE_TRAIN, DEV_INPUT, PROMPT_FILE):
    if not path.exists():
        raise FileNotFoundError(path)

prompt_spec = json.loads(PROMPT_FILE.read_text(encoding="utf-8"))
print(json.dumps({
    "base_model": str(BASE_MODEL),
    "initial_sft_adapter": str(INITIAL_SFT_ADAPTER),
    "rationale_bank": str(RATIONALE_BANK),
    "output_dir": str(OUTPUT_DIR),
    "prompt_id": prompt_spec["prompt_id"],
    "max_seq_length": MAX_SEQ_LENGTH,
    "epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "answer_only_replay_fraction": ANSWER_ONLY_REPLAY_FRACTION,
}, indent=2))

## Prompt contract

This prompt differs from the old answer-only prompt in three essential ways:

1. It tells the model to reason only from **numbered snippets**, matching the annotation evidence IDs.
2. It requires a concrete evidence relation and an explanation of the requested answer slot.
3. It puts the short extractive answer on the final line, allowing the evaluator to recover exactly one `[BE]…[EE]` span.

The answer-only replay prompt is separate and explicit, so a single prompt never has two conflicting output contracts.

In [ ]:
print("RATIONALE PROMPT\n" + "=" * 80)
print(prompt_spec["grounded_rationale_system"])
print("\nANSWER-ONLY REPLAY PROMPT\n" + "=" * 80)
print(prompt_spec["answer_only_replay_system"])

## Build the mixed training set

Every accepted rationale becomes one rationale-SFT example. To reduce format forgetting, a deterministic subset of the same questions is added with the original short answer-only objective. With 1,111 rationale examples, 278 replay examples make answer-only records approximately 20% of the mixed set.

No dev question is used for training.

In [ ]:
from cse_dpo.candidate_bank_class_judge import extract_snippets, candidate_is_extractive
from transformers import AutoTokenizer


def read_jsonl(path: Path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def resources_from_row(row):
    fields = []
    for key, value in row.items():
        match = re.fullmatch(r"input_(\d+)", str(key))
        if match and int(match.group(1)) >= 2 and str(value or "").strip():
            fields.append((int(match.group(1)), str(value).strip()))
    return [value for _, value in sorted(fields)]


def normalized_one_line(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def format_user_context(source_row, snippets):
    snippet_lines = [
        f"Snippet {s['snippet_id']} (PubMed {s['pubmed_id']}): {normalized_one_line(s['text'])}"
        for s in snippets
    ]
    return (
        f"Question: {normalized_one_line(source_row['input_1'])}\n\nPubMed snippets:\n"
        + "\n".join(snippet_lines)
    )


# Load the tokenizer before data export so the notebook can preserve whole snippets
# while enforcing the exact 4,096-token model budget.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, local_files_only=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def chat_length(system, user, assistant_completion=None):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    if assistant_completion is None:
        return len(tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=False
        ))
    messages.append({"role": "assistant", "content": assistant_completion})
    return len(tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False, return_dict=False
    ))


def select_whole_snippets_to_fit(
    source_row,
    system,
    *,
    assistant_completion=None,
    reserve_generation_tokens=0,
    priority_ids=(),
):
    """Keep cited evidence, then add complete distractors in source order.

    Tokenizer truncation is never used. For supervised records, every priority
    snippet is retained. For generation records, no priority label is available,
    so complete snippets are added in their original order.
    """
    all_snippets = extract_snippets(resources_from_row(source_row))
    if not all_snippets:
        raise ValueError(f"{source_row['id']}: no snippets")
    priority_ids = {str(value) for value in priority_ids}
    known_ids = {s["snippet_id"] for s in all_snippets}
    if not priority_ids <= known_ids:
        raise ValueError(f"{source_row['id']}: unknown priority snippet IDs")

    selected_ids = set(priority_ids)

    def selected_in_source_order(extra_id=None):
        ids = selected_ids | ({extra_id} if extra_id else set())
        return [s for s in all_snippets if s["snippet_id"] in ids]

    selected = selected_in_source_order()
    user = format_user_context(source_row, selected)
    if chat_length(system, user, assistant_completion) + reserve_generation_tokens > MAX_SEQ_LENGTH:
        raise ValueError(
            f"{source_row['id']}: required evidence and target exceed {MAX_SEQ_LENGTH} tokens"
        )

    for snippet in all_snippets:
        snippet_id = snippet["snippet_id"]
        if snippet_id in selected_ids:
            continue
        proposed = selected_in_source_order(snippet_id)
        proposed_user = format_user_context(source_row, proposed)
        if chat_length(system, proposed_user, assistant_completion) + reserve_generation_tokens <= MAX_SEQ_LENGTH:
            selected_ids.add(snippet_id)

    selected = selected_in_source_order()
    user = format_user_context(source_row, selected)
    dropped = [s["snippet_id"] for s in all_snippets if s["snippet_id"] not in selected_ids]
    return user, selected, dropped


rationale_rows = read_jsonl(RATIONALE_BANK)
accepted = [row for row in rationale_rows if row.get("status") == "accepted"]
source_rows = json.loads(SOURCE_TRAIN.read_text(encoding="utf-8"))
source_by_id = {str(row["id"]): row for row in source_rows}
dev_rows = json.loads(DEV_INPUT.read_text(encoding="utf-8"))
dev_ids = {str(row["id"]) for row in dev_rows}

assert len(accepted) == 1111, f"Expected 1,111 accepted rationales, found {len(accepted)}"
assert len({row["question_id"] for row in accepted}) == len(accepted)
assert not ({row["question_id"] for row in accepted} & dev_ids), "Train/dev question overlap"
assert set(row["question_id"] for row in accepted) <= set(source_by_id)

rationale_examples = []
answer_replay_candidates = []
for annotation in accepted:
    qid = str(annotation["question_id"])
    source = source_by_id[qid]
    all_snippets = extract_snippets(resources_from_row(source))
    valid_ids = {s["snippet_id"] for s in all_snippets}
    evidence_ids = [str(x) for x in annotation["evidence_ids"]]
    if not evidence_ids or not set(evidence_ids) <= valid_ids:
        raise ValueError(f"{qid}: invalid evidence IDs")
    cited = [s for s in all_snippets if s["snippet_id"] in set(evidence_ids)]
    answer = normalized_one_line(annotation["chosen_answer"])
    reason = normalized_one_line(annotation["reason"])
    if not answer or not reason or "\n" in answer:
        raise ValueError(f"{qid}: empty or multiline answer/reason")
    if not candidate_is_extractive(answer, cited):
        raise ValueError(f"{qid}: answer is not extractive from its cited snippets")
    if any(evidence_id not in reason for evidence_id in evidence_ids):
        raise ValueError(f"{qid}: reason does not name all cited snippet IDs")

    rationale_target = f"Reason: {reason}\nAnswer: [BE]{answer}[EE]"
    user, selected_snippets, dropped_ids = select_whole_snippets_to_fit(
        source,
        prompt_spec["grounded_rationale_system"],
        assistant_completion=rationale_target,
        priority_ids=evidence_ids,
    )
    rationale_examples.append({
        "id": f"{qid}::rationale",
        "question_id": qid,
        "mode": "grounded_rationale",
        "messages": [
            {"role": "system", "content": prompt_spec["grounded_rationale_system"]},
            {"role": "user", "content": user},
            {"role": "assistant", "content": rationale_target},
        ],
        "metadata": {
            "answer": answer,
            "evidence_ids": evidence_ids,
            "requested_answer_type": annotation.get("requested_answer_type"),
            "rationale_rubric_version": annotation.get("rubric_version"),
            "included_snippet_ids": [s["snippet_id"] for s in selected_snippets],
            "dropped_snippet_ids": dropped_ids,
            "context_policy": "all cited snippets retained; whole distractor snippets greedily added in source order; no token truncation",
        },
    })
    answer_replay_candidates.append({
        "id": f"{qid}::answer_only_replay",
        "question_id": qid,
        "mode": "answer_only_replay",
        "messages": [
            {"role": "system", "content": prompt_spec["answer_only_replay_system"]},
            {"role": "user", "content": user},
            {"role": "assistant", "content": f"Answer: [BE]{answer}[EE]"},
        ],
        "metadata": {
            "answer": answer,
            "evidence_ids": evidence_ids,
            "included_snippet_ids": [s["snippet_id"] for s in selected_snippets],
            "dropped_snippet_ids": dropped_ids,
            "context_policy": "same evidence-preserving context as the paired rationale example",
        },
    })

replay_count = round(len(rationale_examples) * ANSWER_ONLY_REPLAY_FRACTION / (1 - ANSWER_ONLY_REPLAY_FRACTION))
rng = random.Random(SEED)
answer_replay_examples = rng.sample(answer_replay_candidates, replay_count)
mixed_examples = rationale_examples + answer_replay_examples
rng.shuffle(mixed_examples)

assert len({row["id"] for row in mixed_examples}) == len(mixed_examples)
assert all(re.fullmatch(r"Reason: .+\nAnswer: \[BE\][^\n]+\[EE\]", row["messages"][2]["content"])
           for row in rationale_examples)
assert all(re.fullmatch(r"Answer: \[BE\][^\n]+\[EE\]", row["messages"][2]["content"])
           for row in answer_replay_examples)
assert all(set(row["metadata"]["evidence_ids"]) <= set(row["metadata"]["included_snippet_ids"])
           for row in mixed_examples)

DATA_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(DATA_DIR / "rationale_sft_accepted1111.jsonl", rationale_examples)
write_jsonl(DATA_DIR / "answer_only_replay.jsonl", answer_replay_examples)
write_jsonl(DATA_DIR / "mixed_train.jsonl", mixed_examples)
pruned_rationale_examples = sum(bool(row["metadata"]["dropped_snippet_ids"]) for row in rationale_examples)
summary = {
    "version": "factoid-grounded-rationale-sft-data-v1",
    "prompt_id": prompt_spec["prompt_id"],
    "seed": SEED,
    "accepted_rationale_examples": len(rationale_examples),
    "accepted_rationale_questions": len({row["question_id"] for row in rationale_examples}),
    "answer_only_replay_examples": len(answer_replay_examples),
    "mixed_examples": len(mixed_examples),
    "actual_answer_only_fraction": len(answer_replay_examples) / len(mixed_examples),
    "rationale_examples_requiring_context_selection": pruned_rationale_examples,
    "max_dropped_snippets": max(len(row["metadata"]["dropped_snippet_ids"]) for row in rationale_examples),
    "context_policy": "preserve all cited snippets; add complete distractors in original order up to 4096 tokens; never truncate a snippet",
    "train_dev_disjoint": True,
    "source_hashes": {
        "rationale_bank": sha256_file(RATIONALE_BANK),
        "source_train": sha256_file(SOURCE_TRAIN),
        "dev_input": sha256_file(DEV_INPUT),
        "prompt_file": sha256_file(PROMPT_FILE),
    },
}
(DATA_DIR / "data_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

In [ ]:
# Review examples before training.
for mode in ("grounded_rationale", "answer_only_replay"):
    example = next(row for row in mixed_examples if row["mode"] == mode)
    print("\n" + "=" * 100)
    print(mode, example["question_id"])
    print("SYSTEM:\n", example["messages"][0]["content"])
    print("USER (first 1,500 characters):\n", example["messages"][1]["content"][:1500])
    print("TARGET:\n", example["messages"][2]["content"])

## Token-length and label-mask preflight

This cell uses the Qwen chat template and refuses to truncate any example. Only assistant completion tokens are supervised; system and user tokens are masked with `-100`. Run this before loading the model.

In [ ]:

def encode_record(row):
    prompt_ids = tokenizer.apply_chat_template(
        row["messages"][:2], tokenize=True, add_generation_prompt=True, return_dict=False
    )
    full_ids = tokenizer.apply_chat_template(
        row["messages"], tokenize=True, add_generation_prompt=False, return_dict=False
    )
    if full_ids[:len(prompt_ids)] != prompt_ids or len(full_ids) <= len(prompt_ids):
        raise ValueError(f"Chat-template prefix mismatch: {row['id']}")
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": [-100] * len(prompt_ids) + full_ids[len(prompt_ids):],
    }


encoded_train = [encode_record(row) for row in mixed_examples]
lengths = [len(row["input_ids"]) for row in encoded_train]
target_lengths = [sum(token != -100 for token in row["labels"]) for row in encoded_train]
too_long = [(mixed_examples[i]["id"], length) for i, length in enumerate(lengths) if length > MAX_SEQ_LENGTH]
if too_long:
    raise ValueError(f"{len(too_long)} examples exceed {MAX_SEQ_LENGTH} tokens; no truncation was applied. First: {too_long[:10]}")

preflight = {
    "examples": len(encoded_train),
    "min_tokens": min(lengths),
    "mean_tokens": sum(lengths) / len(lengths),
    "max_tokens": max(lengths),
    "max_target_tokens": max(target_lengths),
    "max_seq_length": MAX_SEQ_LENGTH,
    "truncated_examples": 0,
    "estimated_optimizer_steps": math.ceil(len(encoded_train) / GRADIENT_ACCUMULATION_STEPS * NUM_TRAIN_EPOCHS),
}
(DATA_DIR / "token_preflight.json").write_text(json.dumps(preflight, indent=2), encoding="utf-8")
print(json.dumps(preflight, indent=2))

## Continue the existing SFT adapter

This cell trains the existing answer-only LoRA adapter in place **in memory**, then writes the result to a new directory. It does not merge the adapter into the base model and does not modify the input adapter.

The default is intentionally conservative: one epoch, `2e-5` learning rate, effective batch size 32. If answer MRR falls after this format-adaptation stage, lower the learning rate or stop at the first saved checkpoint rather than increasing the number of epochs.

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from peft import PeftModel, prepare_model_for_kbit_training

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for training. The data and token preflight cells can run without it.")
if (OUTPUT_DIR / "training_complete.json").exists():
    raise RuntimeError(f"This run is already complete: {OUTPUT_DIR}. Change RUN_NAME for a fresh run.")

set_seed(SEED)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model_kwargs = {
    "local_files_only": True,
    "torch_dtype": dtype,
    "device_map": {"": 0},
    "attn_implementation": "sdpa",
}
if LOAD_IN_4BIT:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
if LOAD_IN_4BIT:
    try:
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )
    except TypeError:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

model = PeftModel.from_pretrained(
    model, INITIAL_SFT_ADAPTER, is_trainable=True, local_files_only=True
)
model.config.use_cache = False
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
model.print_trainable_parameters()


class CompletionOnlyCollator:
    """Pad the full context while computing logits only at supervised positions."""
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        size = max(len(feature["input_ids"]) for feature in features)
        batch = {
            key: torch.tensor(
                [feature[key] + [pad] * (size - len(feature[key])) for feature in features],
                dtype=torch.long,
            )
            for key, pad in (
                ("input_ids", self.pad_token_id),
                ("attention_mask", 0),
                ("labels", -100),
            )
        }
        # This is the same completion-logit projection used by the project's
        # continued evidence-SFT trainer for Qwen2.5.
        shifted = torch.nn.functional.pad(batch["labels"][:, 1:], (0, 1), value=-100)
        positions = torch.where(shifted.ne(-100).any(dim=0))[0]
        if positions.numel() == 0:
            raise ValueError("Batch contains no supervised completion tokens")
        batch["logits_to_keep"] = positions
        batch["shift_labels"] = shifted[:, positions].contiguous()
        return batch


train_dataset = Dataset.from_list(encoded_train)
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,
    logging_steps=1,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit" if LOAD_IN_4BIT else "adamw_torch",
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=CompletionOnlyCollator(tokenizer.pad_token_id),
)
resume_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR)) if CHECKPOINT_DIR.exists() else None
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=False)
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
trainer.save_state()

run_manifest = {
    **summary,
    **preflight,
    "base_model": str(BASE_MODEL.resolve()),
    "initial_sft_adapter": str(INITIAL_SFT_ADAPTER.resolve()),
    "final_adapter": str(FINAL_ADAPTER_DIR.resolve()),
    "epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "load_in_4bit": LOAD_IN_4BIT,
    "train_metrics": train_result.metrics,
}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
(OUTPUT_DIR / "training_complete.json").write_text(
    json.dumps({"status": "complete", "final_adapter": str(FINAL_ADAPTER_DIR)}, indent=2),
    encoding="utf-8",
)
print(json.dumps(run_manifest, indent=2))

## Greedy BioASQ dev evaluation at 4,096 tokens

Run this after training. It evaluates the final adapter with the same rationale prompt, extracts the final `[BE]…[EE]` answer, and invokes the official BioASQ Java scorer. The saved predictions retain the full rationale for inspection.

Set `ADAPTER_TO_EVALUATE = INITIAL_SFT_ADAPTER` if you also want a step-0 comparison under the new prompt.

In [ ]:
import argparse
from src.utility.bioasq_official import evaluate_with_bioasq_java
from src.utility.eval_types import EvalExample

ADAPTER_TO_EVALUATE = FINAL_ADAPTER_DIR
EVAL_NAME = "rationale_sft_final"


def parse_aliases(output):
    aliases = [normalized_one_line(x) for x in re.findall(r"\[BE\]\s*(.*?)\s*\[EE\]", str(output), flags=re.I | re.S)]
    return list(dict.fromkeys(alias for alias in aliases if alias))


def build_dev_records():
    records = []
    for row in dev_rows:
        user, snippets, dropped_ids = select_whole_snippets_to_fit(
            row,
            prompt_spec["grounded_rationale_system"],
            reserve_generation_tokens=MAX_NEW_TOKENS,
        )
        messages = [
            {"role": "system", "content": prompt_spec["grounded_rationale_system"]},
            {"role": "user", "content": user},
        ]
        prompt_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=False
        )
        if len(prompt_ids) + MAX_NEW_TOKENS > MAX_SEQ_LENGTH:
            raise ValueError(
                f"{row['id']}: prompt plus generation budget exceeds {MAX_SEQ_LENGTH}; "
                "evaluation does not silently truncate."
            )
        records.append({
            "question_id": str(row["id"]),
            "question": normalized_one_line(row["input_1"]),
            "gold_aliases": parse_aliases(row["output"]),
            "messages": messages,
            "prompt_ids": prompt_ids,
            "snippet_ids": [s["snippet_id"] for s in snippets],
            "dropped_snippet_ids": dropped_ids,
        })
    return records


def load_eval_model(adapter_path):
    kwargs = {
        "local_files_only": True,
        "torch_dtype": dtype,
        "device_map": {"": 0},
        "attn_implementation": "sdpa",
    }
    if LOAD_IN_4BIT:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype,
        )
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    loaded = PeftModel.from_pretrained(base, adapter_path, is_trainable=False, local_files_only=True)
    loaded.eval()
    loaded.config.use_cache = True
    return loaded


def evaluate_adapter(adapter_path: Path, eval_name: str):
    eval_model = load_eval_model(adapter_path)
    records = build_dev_records()
    predictions = []
    for index, row in enumerate(records, 1):
        tokens = torch.tensor([row["prompt_ids"]], device=eval_model.device)
        with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=dtype):
            generated = eval_model.generate(
                input_ids=tokens,
                attention_mask=torch.ones_like(tokens),
                do_sample=False,
                num_beams=1,
                max_new_tokens=MAX_NEW_TOKENS,
                use_cache=True,
                logits_to_keep=1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        raw = tokenizer.decode(generated[0, tokens.shape[1]:].cpu(), skip_special_tokens=True).strip()
        aliases = parse_aliases(raw)
        answer = aliases[-1] if aliases else ""
        format_valid = bool(re.fullmatch(r"Reason: .+\nAnswer: \[BE\][^\n]+\[EE\]", raw, flags=re.S))
        predictions.append({
            "question_id": row["question_id"],
            "question_type": "factoid",
            "body": row["question"],
            "source_path": str(DEV_INPUT),
            "prediction": f"[BE]{answer}[EE]" if answer else "",
            "answer": answer,
            "raw_prediction": raw,
            "format_valid": format_valid,
        })
        if index % 20 == 0:
            print(f"Generated {index}/{len(records)}", flush=True)

    examples_by_key = {}
    for row in records:
        raw_question = {
            "id": row["question_id"], "type": "factoid", "body": row["question"],
            "exact_answer": [list(row["gold_aliases"])],
        }
        examples_by_key[(row["question_id"], "factoid")] = EvalExample(
            row["question_id"], "factoid", row["question"], "", (),
            f"[BE]{row['gold_aliases'][0]}[EE]", str(DEV_INPUT), raw_question,
        )

    eval_dir = OUTPUT_DIR / "official_generated_eval" / eval_name
    official = evaluate_with_bioasq_java(
        prediction_rows=predictions,
        examples_by_key=examples_by_key,
        model_label=eval_name,
        model_dir=eval_dir,
        args=argparse.Namespace(
            bioasq_java_jar=str(ROOT / "third_party/Evaluation-Measures/flat/BioASQEvaluation/dist/BioASQEvaluation.jar"),
            bioasq_java_heap="512m",
            bioasq_java_version=5,
        ),
        include_per_question=True,
    )
    write_jsonl(eval_dir / "rationale_predictions.jsonl", predictions)
    metrics = official["aggregate"]["by_type"]["factoid"]["metrics"]
    summary_eval = {
        "adapter": str(adapter_path.resolve()),
        "prompt_id": prompt_spec["prompt_id"],
        "max_seq_length": MAX_SEQ_LENGTH,
        "max_new_tokens": MAX_NEW_TOKENS,
        "question_count": len(predictions),
        "questions_requiring_context_selection": sum(bool(row["dropped_snippet_ids"]) for row in records),
        "context_policy": "complete snippets in original order up to 4096 tokens; no token truncation",
        "format_valid_rate": sum(row["format_valid"] for row in predictions) / len(predictions),
        "official_metrics": metrics,
    }
    (eval_dir / "rationale_eval_summary.json").write_text(json.dumps(summary_eval, indent=2), encoding="utf-8")
    print(json.dumps(summary_eval, indent=2))
    return summary_eval, predictions


rationale_eval_summary, rationale_dev_predictions = evaluate_adapter(ADAPTER_TO_EVALUATE, EVAL_NAME)

## Handoff to rationale DPO

Use `adapter_final` from this run as **both**:

- the initial trainable policy adapter; and
- the frozen reference adapter.

Then build DPO completions in the same response contract:

```text
Chosen:  Reason: <accepted gold rationale>\nAnswer: [BE]<C3>[EE]
Rejected: Reason: <plausible C1 rationale>\nAnswer: [BE]<C1>[EE]
```

Do not initialize rationale DPO from the older answer-only SFT adapter, because its reference distribution has not learned the rationale format. Keep dev/test answers out of both rationale SFT and DPO training, and select checkpoints by official dev MRR plus retention, not by training loss alone.